In [ ]:
import glob
import os
import re

# Download Dados por UF

In [ ]:
%pip install pysus

In [ ]:
import pysus  # pyright: ignore[reportMissingImports]

sim = pysus.SIM().load()

sp_cid10 = sim.get_files(
    "CID10",
    uf=[
        "AC",
        "AL",
        "AP",
        "AM",
        "BA",
        "CE",
        "dados_mortalidade",
        "ES",
        "GO",
        "MA",
        "MT",
        "MS",
        "MG",
        "PA",
        "PB",
        "PR",
        "PE",
        "PI",
        "RJ",
        "RN",
        "RS",
        "RO",
        "RR",
        "SC",
        "SP",
        "SE",
        "TO",
    ],
    year=[2023],
)

In [ ]:
sp_cid10

[DOAC2023.dbc,
 DOAL2023.dbc,
 DOAM2023.dbc,
 DOAP2023.dbc,
 DOBA2023.dbc,
 DOCE2023.dbc,
 DODF2023.dbc,
 DOES2023.dbc,
 DOGO2023.dbc,
 DOMA2023.dbc,
 DOMG2023.dbc,
 DOMS2023.dbc,
 DOMT2023.dbc,
 DOPA2023.dbc,
 DOPB2023.dbc,
 DOPE2023.dbc,
 DOPI2023.dbc,
 DOPR2023.dbc,
 DORJ2023.dbc,
 DORN2023.dbc,
 DORO2023.dbc,
 DORR2023.dbc,
 DORS2023.dbc,
 DOSC2023.dbc,
 DOSE2023.dbc,
 DOSP2023.dbc,
 DOTO2023.dbc]

In [ ]:
local_dir = "/br_ms_sim/input/ano=2023"

# for ds in sp_cid10:
#     ds.download(local_dir=local_dir)

In [ ]:
original_names = {
    # Localização
    "CODBAIRES": "codigo_bairro_residencia",
    "CODMUNRES": "id_municipio_residencia",
    "LOCOCOR": "local_ocorrencia",
    "CODBAIOCOR": "codigo_bairro_ocorrencia",
    "CODMUNOCOR": "id_municipio_ocorrencia",
    "CODMUNNATU": "id_municipio_naturalidade",
    "COMUNSVOIM": "id_municipio_svo_iml",
    # Mãe / Materno
    "IDADEMAE": "idade_mae",
    "ESCMAE": "escolaridade_mae",
    "OCUPMAE": "ocupacao_mae",
    "QTDFILVIVO": "quantidade_filhos_vivos",
    "QTDFILMORT": "quantidade_filhos_mortos",
    "GRAVIDEZ": "gravidez",
    "GESTACAO": "gestacao",
    "SEMAGESTAC": "semanas_gestacao",
    "PARTO": "parto",
    "OBITOPARTO": "obito_parto",
    "MORTEPARTO": "morte_parto",
    "OBITOGRAV": "obito_gravidez",
    "OBITOPUERP": "obito_puerperio",
    "PESO": "peso",
    # Assistência / Procedimentos
    "ASSISTMED": "assistencia_medica",
    "EXAME": "exame",
    "CIRURGIA": "cirurgia",
    "NECROPSIA": "necropsia",
    # Causa da morte
    "LINHAA": "linha_a",
    "LINHAB": "linha_b",
    "LINHAC": "linha_c",
    "LINHAD": "linha_d",
    "LINHAII": "linha_ii",
    "CIRCOBITO": "circunstancia_obito",
    "ACIDTRAB": "acidente_trabalho",
    "CAUSABAS_O": "causa_basica_original",
    "CB_PRE": "causa_basica_pre",
    "CAUSAMAT": "causa_materna",
    "ALTCAUSA": "alt_causa",
    # Estabelecimento / Profissional
    "CODESTAB": "codigo_estabelecimento",
    "ESTABDESCR": "descricao_estabelecimento",
    "CRM": "crm",
    "ATESTADO": "atestado",
    "ATESTANTE": "atestante",
    # Datas
    "DTOBITO": "data_obito",
    "DTATESTADO": "data_atestado",
    "DTINVESTIG": "data_investigacao",
    "DTCADASTRO": "data_cadastro",
    "DTRECEBIM": "data_recebimento",
    "DTCADINF": "data_cadastro_informacao",
    "DTCADINV": "data_cadastro_investigacao",
    "DTRECORIGA": "data_recebimento_original_a",
    "DTCONINV": "data_conclusao_investigacao",
    "DTCONCASO": "data_conclusao_caso",
    # Investigação / Status
    "FONTEINV": "fonte_investigacao",
    "TPRESGINFO": "tipo_resgate_informacao",
    "TPNIVELINV": "tipo_nivel_investigador",
    "STDOEPIDEM": "status_do_epidem",
    "STDONOVA": "status_do_nova",
    "STCODIFICA": "status_codificadora",
    "CODIFICADO": "codificado",
    # Escolaridade (2010 / agregada)
    "SERIESCFAL": "serie_escolar_falecido",
    "SERIESCMAE": "serie_escolar_mae",
    "ESC2010": "escolaridade_2010",
    "ESCMAE2010": "escolaridade_mae_2010",
    "ESCFALAGR1": "escolaridade_falecido_2010_agr",
    "ESCMAEAGR1": "escolaridade_mae_2010_agr",
    # Outros
    "HORAOBITO": "hora_obito",
    "DIFDATA": "diferenca_data",
    "NUDIASOBIN": "numero_dias_obito_investigacao",
    "NUDIASOBCO": "numero_dias_obito_ficha",
    "NUDIASINF": "numero_dias_informacao",
    "NUMEROLOTE": "numero_lote",
    "FONTES": "fontes",
    "FONTESINF": "fontes_informacao",
    "TPPOS": "tipo_pos",
    "TIPOBITO": "tipo_obito",
    "TPOBITOCOR": "tipo_obito_ocorrencia",
    "TPMORTEOCO": "tipo_morte_ocorrencia",
    "VERSAOSIST": "versao_sistema",
    "VERSAOSCB": "versao_scb",
    "NATURAL": "naturalidade",
    "DTNASC": "data_nascimento",
    "IDADE": "idade",
    "SEXO": "sexo",
    "RACACOR": "raca_cor",
    "CAUSABAS": "causa_basica",
    "ESTCIV": "estado_civil",
    "ESC": "escolaridade",
    "OCUP": "ocupacao",
    "CONTADOR": "contador",
    "FONTE": "fonte",
}

In [ ]:
import pandas as pd

base_path = "/br_ms_sim/input/ano=2023"

files = glob.glob(os.path.join(base_path, "*.parquet"))


def load_with_uf(f):
    """
    Lê um arquivo parquet e adiciona a coluna 'sigla_uf'
    extraída do nome do arquivo.

    Espera arquivos no formato:
    DOXX2023.parquet
    onde XX é a sigla da UF.
    """

    # Extrai apenas o nome do arquivo
    filename = os.path.basename(f)

    # Regex para capturar UF
    match = re.search(r"DO([A-Z]{2})\d{4}", filename, re.IGNORECASE)

    uf = match.group(1).upper()

    # Lê o arquivo
    dados_mortalidade = pd.read_parquet(f)

    # Adiciona a coluna
    dados_mortalidade["sigla_uf"] = uf

    return dados_mortalidade


# Carrega os arquivos e concatena
dados_mortalidade = pd.concat(
    [load_with_uf(f) for f in files], ignore_index=True
)

# Renomeia colunas
if "original_names" in globals():
    dados_mortalidade = dados_mortalidade.rename(columns=original_names)

dados_mortalidade.head()

,ORIGEM,tipo_obito,data_obito,hora_obito,naturalidade,id_municipio_naturalidade,data_nascimento,idade,sexo,raca_cor,...,tipo_resgate_informacao,tipo_nivel_investigador,numero_dias_informacao,data_cadastro_informacao,morte_parto,data_conclusao_caso,fontes_informacao,alt_causa,contador,sigla_uf
0,1,2,01012023,0610,812,120070,26011941,481,1,4,...,,,,,,,,,2067,AC
1,1,2,01012023,1305,813,130180,20081966,456,2,4,...,,,,,,,,,2146,AC
2,1,2,01012023,0020,812,120030,15061968,454,2,1,...,,,,,,,,,2196,AC
3,1,2,01012023,1430,812,120045,14111965,457,1,4,...,,,,,,,,,2839,AC
4,1,2,01012023,1040,812,120035,12021958,464,2,4,...,,,,,,,,,3066,AC


In [ ]:
colunas_esperadas = [
    "ano",
    "sigla_uf",
    "sequencial_obito",
    "tipo_obito",
    "causa_basica",
    "data_obito",
    "hora_obito",
    "naturalidade",
    "data_nascimento",
    "idade",
    "sexo",
    "raca_cor",
    "estado_civil",
    "escolaridade",
    "ocupacao",
    "codigo_bairro_residencia",
    "id_municipio_residencia",
    "local_ocorrencia",
    "codigo_bairro_ocorrencia",
    "id_municipio_ocorrencia",
    "idade_mae",
    "escolaridade_mae",
    "ocupacao_mae",
    "quantidade_filhos_vivos",
    "quantidade_filhos_mortos",
    "gravidez",
    "gestacao",
    "parto",
    "obito_parto",
    "morte_parto",
    "peso",
    "obito_gravidez",
    "obito_puerperio",
    "assistencia_medica",
    "exame",
    "cirurgia",
    "necropsia",
    "linha_a",
    "linha_b",
    "linha_c",
    "linha_d",
    "linha_ii",
    "circunstancia_obito",
    "acidente_trabalho",
    "fonte",
    "codigo_estabelecimento",
    "atestante",
    "data_atestado",
    "tipo_pos",
    "data_investigacao",
    "causa_basica_original",
    "data_cadastro",
    "fonte_investigacao",
    "data_recebimento",
    "causa_basica_pre",
    "tipo_obito_ocorrencia",
    "tipo_morte_ocorrencia",
    "data_cadastro_informacao",
    "data_cadastro_investigacao",
    "id_municipio_svo_iml",
    "data_recebimento_original",
    "data_recebimento_original_a",
    "causa_materna",
    "status_do_epidem",
    "status_do_nova",
    "serie_escolar_falecido",
    "serie_escolar_mae",
    "escolaridade_2010",
    "escolaridade_mae_2010",
    "escolaridade_falecido_2010_agr",
    "escolaridade_mae_2010_agr",
    "semanas_gestacao",
    "diferenca_data",
    "data_conclusao_investigacao",
    "data_conclusao_caso",
    "numero_dias_obito_investigacao",
    "id_municipio_naturalidade",
    "descricao_estabelecimento",
    "crm",
    "numero_lote",
    "status_codificadora",
    "codificado",
    "versao_sistema",
    "versao_scb",
    "atestado",
    "numero_dias_obito_ficha",
    "fontes",
    "tipo_resgate_informacao",
    "tipo_nivel_investigador",
    "numero_dias_informacao",
    "fontes_informacao",
    "alt_causa",
]

In [ ]:
colunas_faltantes = set(colunas_esperadas) - set(dados_mortalidade.columns)

print(sorted(colunas_faltantes))

[]


In [ ]:
dados_mortalidade["ano"] = 2023
dados_mortalidade["codigo_bairro_ocorrencia"] = None
dados_mortalidade["codigo_bairro_residencia"] = None
dados_mortalidade["crm"] = None
dados_mortalidade["data_recebimento_original"] = None
dados_mortalidade["sequencial_obito"] = None

In [ ]:
dados_mortalidade = dados_mortalidade[colunas_esperadas]

In [ ]:
dados_mortalidade

,ano,sigla_uf,sequencial_obito,tipo_obito,causa_basica,data_obito,hora_obito,naturalidade,data_nascimento,idade,...,versao_sistema,versao_scb,atestado,numero_dias_obito_ficha,fontes,tipo_resgate_informacao,tipo_nivel_investigador,numero_dias_informacao,fontes_informacao,alt_causa
0,2023,AC,None,2,C349,01012023,0610,812,26011941,481,...,3.2.30,3.4,N189/D649*I11 C349 ...,,,,,,,
1,2023,AC,None,2,J159,01012023,1305,813,20081966,456,...,3.2.30,3.4,/J159 U049*I10 ...,,,,,,,
2,2023,AC,None,2,I219,01012023,0020,812,15061968,454,...,3.2.30,3.4,I219/I249*F10 ...,,,,,,,
3,2023,AC,None,2,C169,01012023,1430,812,14111965,457,...,3.2.30,3.3,C787 C788/C169 ...,,,,,,,
4,2023,AC,None,2,B342,01012023,1040,812,12021958,464,...,3.2.00,3.4,J960/J159/B342 U071*I10 I500 ...,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465605,2023,TO,None,2,X950,31122023,1845,817,01091992,431,...,3.2.30,3.4,S069/X950 ...,,,,,,,
1465606,2023,TO,None,2,E142,31122023,1720,821,02051969,454,...,3.2.30,3.4,J969/J189/R568/N189*I10 E149 ...,,,,,,,
1465607,2023,TO,None,2,C109,31122023,1205,852,01111964,459,...,3.2.30,3.4,J960/J980/C109 ...,,,,,,,
1465608,2023,TO,None,2,I619,31122023,0200,821,27011977,446,...,3.2.30,3.4,R578/J189/I619/I10*E149 ...,162,,,E,,,


In [ ]:
from pathlib import Path

output = Path("/br_ms_sim/output/ano=2023")

Path(output).mkdir(parents=True, exist_ok=True)

In [ ]:
parts = dados_mortalidade.groupby(["ano", "sigla_uf"])

In [ ]:
for uf, part in dados_mortalidade.groupby("sigla_uf"):
    dir_path = output / f"sigla_uf={uf}"
    dir_path.mkdir(parents=True, exist_ok=True)

    part.drop(columns=["ano", "sigla_uf"]).to_csv(
        dir_path / "microdados.csv", index=False
    )

Arquitetura e Dicionário

In [ ]:
# Quais colunas tem dicionário

dados_mortalidade = pd.read_csv(
    "/br_ms_sim/staging_br_ms_sim_dicionario_dicionario.csv"
)

# filtrar id_tabela == 'microdados' e pegar colunas únicas
colunas_unicas = (
    dados_mortalidade.loc[
        dados_mortalidade["id_tabela"] == "microdados", "coluna"
    ]
    .dropna()
    .unique()
)

colunas_unicas = colunas_unicas.tolist()

arquitetura = pd.read_csv("/[arquitetura] microdados.csv")

arquitetura.loc[
    arquitetura["name"].isin(colunas_unicas), "covered_by_dictionary"
] = "yes"

arquitetura.loc[
    ~arquitetura["name"].isin(colunas_unicas), "covered_by_dictionary"
] = "no"

arquitetura.to_csv("arquitetura.csv", index=False)

/tmp/ipykernel_239/707346965.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  arquitetura.loc[
